# DocStruct — section-boundary agreement, full PMC corpus (Colab T4)

Upload this notebook, set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Nothing else to do. Every cell is idempotent.

## Why this notebook exists

The 2026-08-13 run produced a complete seven-tool table — and scored **24 documents, not 126**,
because that session only ever had 24 PMC PDFs on disk. `score_sections.py` skips a document
whose PDF is missing, silently, so a short corpus looks exactly like a finished run.
**This notebook makes the corpus size a checked precondition instead of a surprise** (section 5).

## It is meant to be run more than once

Free Colab reclaims sessions without warning. Everything expensive lives on Drive: the corpus,
the model weights, the detector cache, the PDF text spines and the per-tool section checkpoints.
**If the session dies, reopen and Run all again** — every stage skips what is already done.

## What it produces

1. **`section_scores.md` / `.json`** — Pk, WindowDiff and straddle rate for seven chunkers
   against the publisher's own JATS section boundaries. Lower is better; 0.0 is perfect.
2. **`section_reachability.json`** — the ceiling, over *the same documents that got scored*.
   A gold boundary that cannot be found in the PDF's own text cannot be scored against, and a
   Pk that quietly skipped a third of them would still look like a result.

No retriever and no relevance rule are involved, so none of the size-bias in
`memory/relevance-modes.md` applies here. That is the point of this metric.

## 1. GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "NO GPU - Runtime > Change runtime type > T4 GPU, then Run all again."
print('GPU ok:', torch.cuda.get_device_name(0))

# Record the Pillow this kernel has ALREADY loaded, before anything pip-installs over it.
# Colab imports PIL at startup, and PIL._imaging is a compiled C extension: once it is in
# the process it cannot be unloaded -- not by deleting sys.modules entries, not by
# reimporting. If pip later upgrades the Python files underneath it, every PIL import dies
# with "The _imaging extension was built for another version of Pillow" and takes
# ultralytics down with it. Pinning back to this exact version after the install keeps the
# files and the loaded extension in agreement, which is what avoids a runtime restart
# mid-"Run all" -- and a restart cannot be done from inside Run all.
import PIL
PRELOADED_PILLOW = PIL.__version__
print('pillow already loaded in this kernel:', PRELOADED_PILLOW)

## 2. Drive — every expensive artefact lives here

`BENCH` is the single directory this notebook owns. Deleting it resets everything; leaving it
alone is what makes a re-run resume.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BENCH    = '/content/drive/MyDrive/docstruct_bench'
CORPORA  = BENCH + '/corpora'        # fetched PDFs + JATS, so a dead session never refetches
CACHE    = BENCH + '/.bench_cache'   # detector proposals
DOTCACHE = BENCH + '/.cache'         # PDF text spines + section-scorer checkpoints
WEIGHTS  = BENCH + '/weights'
REPORTS  = BENCH + '/reports'
for d in (BENCH, CORPORA, CACHE, DOTCACHE, WEIGHTS, REPORTS, CORPORA + '/pmc'):
    os.makedirs(d, exist_ok=True)
print('\n'.join((BENCH, CORPORA, CACHE, DOTCACHE, WEIGHTS, REPORTS)))

## 3. Repo and dependencies

**Ignore pip's dependency-conflict warnings** about `jedi`, `requests`/`google-colab`,
`opentelemetry`/`google-adk` and the `huggingface-hub` `inference` extra. Pip reports on the
whole environment, including Colab preinstalls this benchmark never imports. Only a traceback
matters.

The one that *is* fatal is Pillow — section 1 recorded the version this kernel loaded at
startup, and the cell below pins it straight back after the install.

In [ ]:
%cd /content
if not os.path.exists('/content/DocStruct/.git'):
    !git clone -q -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git pull -q --ff-only 2>/dev/null || echo '(could not fast-forward; keeping what is here)'
!git log --oneline -1

In [ ]:
!pip install -q -e ".[all,benchmark-heavy]" unstructured-inference pyarrow \
   llama-index-core llama-index-embeddings-huggingface

# Put Pillow back to the version this kernel loaded at startup (see section 1).
!pip install -q "pillow=={PRELOADED_PILLOW}"

import PIL
from PIL import Image
Image.new('RGB', (4, 4))
assert PIL.__version__ == PRELOADED_PILLOW, (
    f'pillow is {PIL.__version__}, kernel loaded {PRELOADED_PILLOW} -- the C extension and the '
    'Python files will disagree on the next import. Re-run this cell.')
print('pillow', PIL.__version__, 'ok (core and files agree)')

In [ ]:
# Ultralytics ships a top-level `tests` package that shadows ours, and importing YOLO is what
# puts it on the path. Confirm the real PIL survives an ultralytics import before the long run
# rather than 40 minutes into it.
from ultralytics import YOLO
from PIL import Image
Image.new('RGB', (4, 4))
print('ultralytics imported, PIL still alive')

### Point the caches at Drive

`.cache/` holds the PDF text spines and the per-tool section checkpoints. Symlinking the whole
directory means a killed run resumes without any flag being passed.

In [ ]:
import os
if not os.path.islink('/content/DocStruct/.cache'):
    !rm -rf /content/DocStruct/.cache
    os.symlink(DOTCACHE, '/content/DocStruct/.cache')
print('.cache ->', os.path.realpath('/content/DocStruct/.cache'))

In [ ]:
# Which adapters actually imported. get_adapters() swallows import errors and drops anything
# whose available() is False, so a missing dependency silently shrinks the leaderboard instead
# of failing. Read the MISSING line before the long run.
from docstruct.eval.adapters import get_adapters
WANT = ['docstruct', 'docstruct_geo', 'langchain', 'pymupdf4llm',
        'unstructured', 'llamaindex', 'llamaindex_semantic']
got = get_adapters(names=WANT, weights=None)
print('available:', sorted(got))
print('MISSING  :', sorted(set(WANT) - set(got)))
TOOLS = ','.join(n for n in WANT if n in got)
print('TOOLS    :', TOOLS)
assert len(got) >= 6, 'too many adapters failed to import; the table would be incomparable'

## 4. Weights (cached on Drive)

In [ ]:
!mkdir -p weights
W = 'weights/yolov8m-doclaynet.pt'
if not os.path.exists(WEIGHTS + '/yolov8m-doclaynet.pt'):
    !wget -q --show-progress -O "{WEIGHTS}/yolov8m-doclaynet.pt" \
       https://huggingface.co/hantian/yolo-doclaynet/resolve/main/yolov8m-doclaynet.pt
!cp -n "{WEIGHTS}/yolov8m-doclaynet.pt" {W}
!ls -la weights/

# Confirm YOLO lands on the GPU. Nothing in docstruct/ sets a device; ultralytics and
# sentence-transformers auto-select CUDA. If this prints cpu, stop and say so -- hybrid
# docstruct on CPU is hours, not minutes.
from docstruct.model.detector import ModelDetector
_m = ModelDetector(weights=W)._ensure_model()
print('YOLO device:', next(_m.model.parameters()).device)

## 5. Corpus — and the check the last run did not have

133 open-access papers across 7 journals, each fetched **with the publisher's JATS XML**. The
XML is the gold; the PDF is what the chunkers see.

Drive is the source of truth so a dead session never refetches. NCBI rate-limits and drops
connections, and a single `fetch_pmc.py` pass can come back short — which is exactly how the
previous run ended up scoring 24 documents. So: fetch in a loop until the count stops growing,
then **assert the corpus is large enough before spending GPU time on it**.

In [ ]:
MIN_DOCS = 100   # of a possible ~133. Below this, do not spend the GPU hours.
PER_JOURNAL = 20

import glob, subprocess
!mkdir -p data/pmc
!cp -n "{CORPORA}/pmc/"* data/pmc/ 2>/dev/null || true

def paired():
    """Docs with BOTH a PDF and its JATS -- either alone is useless."""
    pdfs = {os.path.basename(p)[:-4] for p in glob.glob('data/pmc/*.pdf')}
    xmls = {os.path.basename(p)[:-4] for p in glob.glob('data/pmc/*.xml')}
    return pdfs & xmls

prev = -1
for attempt in range(4):
    have = len(paired())
    print(f'-- pass {attempt}: {have} paired documents on disk')
    if have >= MIN_DOCS or have == prev:
        break
    prev = have
    subprocess.run(['python', 'scripts/fetch_pmc.py', '--per-journal', str(PER_JOURNAL)])
    !cp -n data/pmc/* "{CORPORA}/pmc/" 2>/dev/null || true

!cp -n data/pmc/* "{CORPORA}/pmc/" 2>/dev/null || true
N_PAIRED = len(paired())
print(f'\n{N_PAIRED} paired PDF+JATS documents')

In [ ]:
assert N_PAIRED >= MIN_DOCS, (
    f'only {N_PAIRED} paired documents (want >= {MIN_DOCS}). NCBI is refusing or throttling.\n'
    'score_sections.py skips a missing PDF silently, so running now would produce a\n'
    'complete-looking table over a fraction of the corpus -- which is the exact defect\n'
    'this notebook exists to prevent.\n\n'
    'Fix: copy your laptop\'s data/pmc/ (PDFs + XMLs) into Drive at\n'
    f'  {CORPORA}/pmc/\n'
    'and re-run this section. Or lower MIN_DOCS deliberately and record the real N\n'
    'next to every number the run produces.')

# JATS -> section gold. Rebuild every time: it is seconds, and it must describe the XML that
# is actually on disk now, not the XML that was there when the session started.
!python scripts/build_jats_gold.py

import json
gold = json.load(open('data/qa/pmc_sections.json'))
print(f'gold: {len(gold)} documents, {sum(len(v) for v in gold.values())} sections')
assert len(gold) >= MIN_DOCS, f'gold covers only {len(gold)} documents'

## 6. Smoke — three papers through the exact path section 8 uses

Smoke what is about to run, with the same flags. Five failures in an earlier session were
invisible to a green test suite and only surfaced by running the real CLI: a missing
`unstructured-inference`, `QAItem(**d)` rejecting external gold, an argparse choice that did
not exist, a silently-unapplied adapter patch, and ultralytics shadowing our `tests` package.

This uses `docstruct` (hybrid, the slow one) rather than a cheap tool, because the GPU path is
the part that has never been smoked.

In [ ]:
!python scripts/section_reachability.py --limit 3 --out /content/smoke_reach.json
!python scripts/score_sections.py --limit 3 --tools docstruct,docstruct_geo,langchain \
    --weights {W} --cache-dir "{CACHE}" \
    --ckpt-dir /content/smoke_ckpt --out /content/smoke_sections.json \
    --report-md /content/smoke_sections.md

In [ ]:
import json
r = json.load(open('/content/smoke_reach.json'))
s = json.load(open('/content/smoke_sections.json'))
print('reachability ceiling (body):', r.get('body_ceiling_pct'), '%')
for name, v in s['results'].items():
    print(f"  {name:16} WindowDiff={v['windowdiff']}  Pk={v['pk']}  docs={v['n_docs']}  errors={v['errors']}")

assert s['results'], 'no tool scored a single document'
assert all(v['n_docs'] > 0 for v in s['results'].values()), 'a tool scored zero documents'
assert 'docstruct' in s['results'], 'the hybrid GPU path did not run'
assert r.get('body_ceiling_pct', 0) > 50, 'gold is mostly unreachable in the PDF text'
print('\nsmoke ok')

## 7. The ceiling, over the full corpus

Runs first, and on the same documents the scores will cover. Scores mean nothing read against
an assumed 100%: the body ceiling has measured 84.5% over 126 documents.

In [ ]:
!python scripts/section_reachability.py
!cp -f reports/section_reachability.json "{REPORTS}/" 2>/dev/null || true

import json
reach = json.load(open('reports/section_reachability.json'))
print({k: v for k, v in reach.items() if k != 'per_doc'})

## 8. Section-boundary agreement — the run

Does a chunker split where the document splits? Pk and WindowDiff against the publisher's own
JATS boundaries — **lower is better**. Unlike section *paths*, this is a real comparison:
every chunker has boundaries.

Budget: the 24-document run took 312 s for hybrid `docstruct` and 132 s for `docstruct_geo`.
Scaled to ~126 documents that is roughly **1.5–2 h wall-clock for all seven tools**. Checkpoints
live in `.cache/` on Drive, so a reclaimed session resumes rather than restarts — if it dies,
just Run all again.

In [ ]:
!python scripts/score_sections.py --tools {TOOLS} --weights {W} --cache-dir "{CACHE}"
!cp -f reports/section_scores.md reports/section_scores.json "{REPORTS}/" 2>/dev/null || true

In [ ]:
# The check the last run was missing: did the table actually cover the corpus, or did
# score_sections.py quietly skip most of it? n_docs per tool is the number that goes in the
# paper caption -- read it here, not after the session is gone.
import json
sc = json.load(open('reports/section_scores.json'))
scored = {t: v['n_docs'] for t, v in sc['results'].items()}
print('gold documents      :', len(gold))
print('reachability covered:', reach['n_docs'])
print('scored per tool     :', scored)

worst = min(scored.values())
if worst < 0.8 * reach['n_docs']:
    print(f'\nWARNING: a tool scored only {worst} of {reach["n_docs"]} reachable documents.')
    print('Check its `errors` count before quoting its row -- unstructured errored on 6/24 last time.')
else:
    print('\ncoverage ok -- every tool scored most of the reachable corpus')

from IPython.display import Markdown, display
display(Markdown(open('reports/section_scores.md').read()))

## 9. Collect everything

In [ ]:
!cd reports && zip -q -r /content/pmc_sections_results.zip \
    section_scores.md section_scores.json section_reachability.json
!cp -f /content/pmc_sections_results.zip "{BENCH}/" 2>/dev/null || true
!ls -la "{REPORTS}/"
try:
    from google.colab import files
    files.download('/content/pmc_sections_results.zip')
except Exception as e:
    print('browser download skipped:', e)
    print('the zip is on Drive at', BENCH)

---
## Back on the laptop

Unpack into `reports/`, commit the JSONs — they are the paper's evidence — and then:

1. **Check `n_docs` agrees between `section_scores.json` and `section_reachability.json`.**
   The 2026-08-13 run did not (24 vs 126) and that is the whole reason this notebook exists.
   If they now match, delete `reports/section_reachability_colab24.json` and drop the
   subset caveat from `memory/results.md` and `notes.md` Stage 20.
2. **Read the ceiling before the scores.** ~84.5% of body sections are locatable; a score is
   read against that, never against 100%.
3. **Report Pk and WindowDiff together, always.** WindowDiff counts boundaries per window and
   punishes over-segmentation; Pk forgives it. langchain (74 chunks) and unstructured (92)
   against a ~21-section gold are the case in point — quoting one metric alone is quotable in
   either direction. `memory/metrics-justification.md` has the wording.
4. **Straddle rate is descriptive, not an error term.** 57.4% of gold sections are shorter than
   `MIN_CHUNK_TOKENS`, so merging them is the design working.
5. **Treat the table as provisional.** That metric found six defects in itself before producing
   a usable number — LaTeX-polluted gold, a collapsing aligner, a word-token spine,
   forward-only poisoning, punctuation sensitivity, and monotonicity applied to chunks. Five of
   the six made a competitor look worse than it is. The retrieval numbers have months of
   scrutiny; these have one session.